In [0]:
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
import os

print("Running Data Pipeline")

# Load Cleaned Silver Data
df_silver = spark.table("workspace.ecommerce.user_features_silver")

# Feature Engineering & Target Setup
df_features = df_silver.select("user_id", "total_views", "total_cart_adds")
df_labels = df_silver.select("user_id", "total_purchases").withColumn("label", when(col("total_purchases") > 0, 1).otherwise(0)).drop("total_purchases")

# Final Model-Ready Dataset
final_data = df_features.join(df_labels, on="user_id", how="inner")

In [0]:
import mlflow
import mlflow.spark

print("Running ML Pipeline & Saving Production Model")

assembler = VectorAssembler(inputCols=["total_views", "total_cart_adds"], outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label")
production_pipeline = Pipeline(stages=[assembler, lr])

mlflow_temp_path = "/Volumes/workspace/ecommerce/ecommerce_data/mlflow_tmp_final"
dbutils.fs.mkdirs(mlflow_temp_path)
os.environ["MLFLOW_DFS_TMP"] = mlflow_temp_path

# Training & Saving the Model
with mlflow.start_run(run_name="Day14_Final_Production_Run"):
    final_model = production_pipeline.fit(final_data)
    
    mlflow.spark.log_model(final_model, "production_buyer_prediction_model")

In [0]:
from pyspark.ml.functions import vector_to_array

print("Batch Inference & Gold Layer Generation")

# Generate Predictions using the Final Model
predictions_df = final_model.transform(final_data)

# Format the Output
production_output = predictions_df.select(
    "user_id",
    col("label").alias("actual_purchase_status"),
    col("prediction").cast("int").alias("predicted_to_buy"),
    vector_to_array(col("probability"))[1].alias("buying_probability")
)

# Save to Final Gold Table
gold_table_name = "workspace.ecommerce.final_production_gold"
production_output.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)

print(f"Output saved to Gold Table: {gold_table_name}")

display(production_output.filter((col("predicted_to_buy") == 1) & (col("actual_purchase_status") == 0)).orderBy(col("buying_probability").desc()).limit(10))